# Reproduce CheXNet: Explore Predictions

## Import other modules and pandas

In [2]:
import visualize_prediction as V

import pandas as pd

#suppress pytorch warnings about source code changes
import warnings
warnings.filterwarnings('ignore')

## Settings for review
We can examine individual results in more detail, seeing probabilities of disease for test images. 

We get you started with a small number of the images from the large NIH dataset. 

To explore the full dataset, [download images from NIH (large, ~40gb compressed)](https://nihcc.app.box.com/v/ChestXray-NIHCC), extract all tar.gz files to a single folder, place that path  below and set STARTER_IMAGES=False

In [3]:
# STARTER_IMAGES=True
# PATH_TO_IMAGES = "starter_images/"

STARTER_IMAGES=False
PATH_TO_IMAGES = "sample_dataset/"

Load pretrained model (part of cloned repo; should not need to change path unless you want to point to one you retrained)

In [4]:
PATH_TO_MODEL = "pretrained/checkpoint"

In [5]:
LABEL="Pneumonia"

It's more interesting when initially exploring to see cases positive for pathology of interest:

In [6]:
POSITIVE_FINDINGS_ONLY=False

## Load data

This loads up dataloader and model (note: only test images not used for model training are loaded).

In [7]:
import torch
checkpoint = torch.load(PATH_TO_MODEL, map_location=lambda storage, loc: storage)
model = checkpoint['model']
del checkpoint
model.cpu()

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [8]:
import importlib
import cxr_dataset   # your python file mymodule.py

importlib.reload(cxr_dataset)

<module 'cxr_dataset' from 'e:\\Fall_2025\\ENEE739\\Project\\reproduce-chexnet\\cxr_dataset.py'>

In [9]:
import cxr_dataset as CXR
from torchvision import datasets, models, transforms

In [10]:
checkpoint = torch.load(PATH_TO_MODEL, map_location=lambda storage, loc: storage)
model = checkpoint['model']
del checkpoint
model.cpu()

# build dataloader on test
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

data_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

dataset = CXR.CXRDataset(
    path_to_images=PATH_TO_IMAGES,
    path_to_labels="sample_dataset/labels.csv",
    transform=data_transform,
    starter_images=STARTER_IMAGES)

dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=1, shuffle=False, num_workers=1)

In [11]:
len(dataloader)

3110

In [12]:
class densenet_last_layer(torch.nn.Module):
    def __init__(self, model):
        super(densenet_last_layer, self).__init__()
        self.features = torch.nn.Sequential(
            *list(model.children())[:-1]
        )

    def forward(self, x):
        x = self.features(x)
        x_relu = torch.nn.functional.relu(x, inplace=True)
        return x, x_relu

In [13]:
import numpy as np
images, labels, name = next(iter(dataloader))
print(images.shape, labels, name)
x = images
model_cam = densenet_last_layer(model)
y, y_relu = model_cam(x)
pooled = torch.nn.functional.adaptive_avg_pool2d(y, (1, 1))
y.shape, pooled.shape

torch.Size([1, 3, 224, 224]) tensor([[1]], dtype=torch.int32) ('00005567_000.png',)


(torch.Size([1, 1024, 7, 7]), torch.Size([1, 1024, 1, 1]))

In [ ]:
import torch
import numpy as np
import pandas as pd

model_cam = densenet_last_layer(model)
model_cam.eval()

all_features = []
all_labels = []
all_names = []

with torch.no_grad():
    for images, labels, names in dataloader:
        
        # Forward pass to get feature maps
        y, y_relu = model_cam(images)

        # y shape: (B, 1024, 7, 7)
        pooled = torch.nn.functional.adaptive_avg_pool2d(y, (1, 1))  
        # pooled shape: (B, 1024, 1, 1)

        pooled = pooled.view(pooled.size(0), -1)  
        # now shape → (B, 1024)

        # Convert to numpy
        pooled_np = pooled.cpu().numpy()

        all_features.append(pooled_np)
        all_labels.append(labels.cpu().numpy())
        all_names.extend(names)   # list of strings

# Combine all batches
all_features = np.vstack(all_features)
all_labels = np.concatenate(all_labels)

print("Final feature matrix:", all_features.shape)   # (N, 1024)



In [ ]:
df = pd.DataFrame(all_features)
df['label'] = all_labels
df['name'] = all_names

df.to_csv("chexnet_features.csv", index=False)
print("Saved to chexnet_features.csv")

Saved to chexnet_features.csv


In [15]:
import pandas as pd
import numpy as np

# ------------------------------------------
# Load files
# ------------------------------------------
feat_df = pd.read_csv("chexnet_features.csv")
train_meta = pd.read_csv("documents/41598_2021_87762_MOESM1_ESM.csv")

# Make sure image name columns match
# Assume feat_df["name"] and train_meta["Image Index"] contain filenames
train_names = set(train_meta["Image Index"].astype(str))

# Add new column
feat_df["fold"] = "unassigned"

# ------------------------------------------
# Assign train fold from metadata file
# ------------------------------------------
feat_df.loc[feat_df["name"].isin(train_names), "fold"] = "train"

# ------------------------------------------
# Split remaining data according to conditions
# ------------------------------------------
remaining = feat_df[feat_df["fold"] == "unassigned"]

# Positive / Negative masks
pos = remaining[remaining["label"] == 1]
neg = remaining[remaining["label"] == 0]

# Required counts
POS_TRAIN, POS_VAL, POS_TEST = 82, 250, 305
NEG_TRAIN, NEG_VAL, NEG_TEST = 1618, 250, 305

# Shuffle once
pos = pos.sample(frac=1, random_state=42)
neg = neg.sample(frac=1, random_state=42)

# Select splits
pos_train = pos.iloc[:POS_TRAIN]
pos_val   = pos.iloc[POS_TRAIN : POS_TRAIN + POS_VAL]
pos_test  = pos.iloc[POS_TRAIN + POS_VAL : POS_TRAIN + POS_VAL + POS_TEST]

neg_train = neg.iloc[:NEG_TRAIN]
neg_val   = neg.iloc[NEG_TRAIN : NEG_TRAIN + NEG_VAL]
neg_test  = neg.iloc[NEG_TRAIN + NEG_VAL : NEG_TRAIN + NEG_VAL + NEG_TEST]

# Assign folds back into feat_df
feat_df.loc[pos_train.index, "fold"] = "train"
feat_df.loc[pos_val.index,   "fold"] = "val"
feat_df.loc[pos_test.index,  "fold"] = "test"

feat_df.loc[neg_train.index, "fold"] = "train"
feat_df.loc[neg_val.index,   "fold"] = "val"
feat_df.loc[neg_test.index,  "fold"] = "test"

# ------------------------------------------
# Check distributions
# ------------------------------------------
print("\n==== FINAL DISTRIBUTION ====\n")

for split in ["train", "val", "test"]:
    df_split = feat_df[feat_df["fold"] == split]
    pos_count = (df_split["label"] == 1).sum()
    neg_count = (df_split["label"] == 0).sum()
    print(f"{split.upper():5} → Pos:{pos_count:4} | Neg:{neg_count:4} | Total:{len(df_split)}")

total_pos = (feat_df["label"] == 1).sum()
total_neg = (feat_df["label"] == 0).sum()
print("\nOverall:", "Pos:", total_pos, "Neg:", total_neg, "Total:", len(feat_df))


# ------------------------------------------
# Save new CSV
# ------------------------------------------
feat_df.to_csv("chexnet_features_with_folds.csv", index=False)
print("\nSaved as chexnet_features_with_folds.csv")



==== FINAL DISTRIBUTION ====

TRAIN → Pos: 200 | Neg:1800 | Total:2000
VAL   → Pos: 250 | Neg: 250 | Total:500
TEST  → Pos: 305 | Neg: 305 | Total:610

Overall: Pos: 755 Neg: 2355 Total: 3110

Saved as chexnet_features_with_folds.csv


Additional dataset

In [1]:
import visualize_prediction as V

import pandas as pd

#suppress pytorch warnings about source code changes
import warnings
warnings.filterwarnings('ignore')

In [2]:
STARTER_IMAGES=False
PATH_TO_IMAGES = "additional_dataset/"

In [3]:
PATH_TO_MODEL = "pretrained/checkpoint"
LABEL="Pneumonia"
POSITIVE_FINDINGS_ONLY=False

In [4]:
import torch
checkpoint = torch.load(PATH_TO_MODEL, map_location=lambda storage, loc: storage)
model = checkpoint['model']
del checkpoint
model.cpu()

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [5]:
import importlib
import cxr_dataset   # your python file mymodule.py

importlib.reload(cxr_dataset)

<module 'cxr_dataset' from 'e:\\Fall_2025\\ENEE739\\Project\\reproduce-chexnet\\cxr_dataset.py'>

In [6]:
import cxr_dataset as CXR
from torchvision import datasets, models, transforms

In [ ]:
checkpoint = torch.load(PATH_TO_MODEL, map_location=lambda storage, loc: storage)
model = checkpoint['model']
del checkpoint
model.cpu()

# build dataloader on test
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

data_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

dataset = CXR.CXRDataset(
    path_to_images=PATH_TO_IMAGES,
    path_to_labels="additional_dataset/extra_train_500.csv",
    transform=data_transform,
    starter_images=STARTER_IMAGES)

dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=1, shuffle=False, num_workers=1)

In [8]:
len(dataloader)

500

In [9]:
print(PATH_TO_IMAGES)

additional_dataset/


In [10]:
class densenet_last_layer(torch.nn.Module):
    def __init__(self, model):
        super(densenet_last_layer, self).__init__()
        self.features = torch.nn.Sequential(
            *list(model.children())[:-1]
        )

    def forward(self, x):
        x = self.features(x)
        x_relu = torch.nn.functional.relu(x, inplace=True)
        return x, x_relu

In [11]:
import numpy as np
images, labels, name = next(iter(dataloader))
print(images.shape, labels, name)
x = images
model_cam = densenet_last_layer(model)
y, y_relu = model_cam(x)
pooled = torch.nn.functional.adaptive_avg_pool2d(y, (1, 1))
y.shape, pooled.shape

torch.Size([1, 3, 224, 224]) tensor([[0]], dtype=torch.int32) ('00028897_020.png',)


(torch.Size([1, 1024, 7, 7]), torch.Size([1, 1024, 1, 1]))

In [12]:
import torch
import numpy as np
import pandas as pd

model_cam = densenet_last_layer(model)
model_cam.eval()

all_features = []
all_labels = []
all_names = []

with torch.no_grad():
    for images, labels, names in dataloader:
        
        # Forward pass to get feature maps
        y, y_relu = model_cam(images)

        # y shape: (B, 1024, 7, 7)
        pooled = torch.nn.functional.adaptive_avg_pool2d(y, (1, 1))  
        # pooled shape: (B, 1024, 1, 1)

        pooled = pooled.view(pooled.size(0), -1)  
        # now shape → (B, 1024)

        # Convert to numpy
        pooled_np = pooled.cpu().numpy()

        all_features.append(pooled_np)
        all_labels.append(labels.cpu().numpy())
        all_names.extend(names)   # list of strings

# Combine all batches
all_features = np.vstack(all_features)
all_labels = np.concatenate(all_labels)

print("Final feature matrix:", all_features.shape)   # (N, 1024)



Final feature matrix: (500, 1024)


In [13]:
df = pd.DataFrame(all_features)
df['label'] = all_labels
df['name'] = all_names

df.to_csv("./features/additional_chexnet_features.csv", index=False)
print("Saved to additional_chexnet_features.csv")


Saved to additional_chexnet_features.csv


In [15]:
import pandas as pd
import numpy as np

# ------------------------------------------
# Load files
# ------------------------------------------
feat_df = pd.read_csv("./features/additional_chexnet_features.csv")
# train_meta = pd.read_csv("documents/41598_2021_87762_MOESM1_ESM.csv")

# Make sure image name columns match
# Assume feat_df["name"] and train_meta["Image Index"] contain filenames
# train_names = set(train_meta["Image Index"].astype(str))

# Add new column
feat_df["fold"] = "unassigned"

# ------------------------------------------
# Assign train fold from metadata file
# ------------------------------------------
# feat_df.loc[feat_df["name"].isin(train_names), "fold"] = "train"

# ------------------------------------------
# Split remaining data according to conditions
# ------------------------------------------
# remaining = feat_df[feat_df["fold"] == "unassigned"]

# Positive / Negative masks
pos = feat_df[feat_df["label"] == 1]
neg = feat_df[feat_df["label"] == 0]

# Required counts
# POS_TRAIN, POS_VAL, POS_TEST = 82, 250, 305
# NEG_TRAIN, NEG_VAL, NEG_TEST = 1618, 250, 305

# Shuffle once
pos = pos.sample(frac=1, random_state=42)
neg = neg.sample(frac=1, random_state=42)

# Select splits
# pos_train = pos.iloc[:POS_TRAIN]
# pos_val   = pos.iloc[POS_TRAIN : POS_TRAIN + POS_VAL]
# pos_test  = pos.iloc[POS_TRAIN + POS_VAL : POS_TRAIN + POS_VAL + POS_TEST]

# neg_train = neg.iloc[:NEG_TRAIN]
# neg_val   = neg.iloc[NEG_TRAIN : NEG_TRAIN + NEG_VAL]
# neg_test  = neg.iloc[NEG_TRAIN + NEG_VAL : NEG_TRAIN + NEG_VAL + NEG_TEST]

# Assign folds back into feat_df
# feat_df.loc[pos_train.index, "fold"] = "train"
# feat_df.loc[pos_val.index,   "fold"] = "val"
feat_df.loc[pos.index,  "fold"] = "test"

# feat_df.loc[neg_train.index, "fold"] = "train"
# feat_df.loc[neg_val.index,   "fold"] = "val"
feat_df.loc[neg.index,  "fold"] = "test"

# ------------------------------------------
# Check distributions
# ------------------------------------------
# print("\n==== FINAL DISTRIBUTION ====\n")

# for split in ["train", "val", "test"]:
#     df_split = feat_df[feat_df["fold"] == split]
#     pos_count = (df_split["label"] == 1).sum()
#     neg_count = (df_split["label"] == 0).sum()
#     print(f"{split.upper():5} → Pos:{pos_count:4} | Neg:{neg_count:4} | Total:{len(df_split)}")

# total_pos = (feat_df["label"] == 1).sum()
# total_neg = (feat_df["label"] == 0).sum()
# print("\nOverall:", "Pos:", total_pos, "Neg:", total_neg, "Total:", len(feat_df))


# ------------------------------------------
# Save new CSV
# ------------------------------------------
feat_df.to_csv("./features/additional_chexnet_features_with_folds.csv", index=False)
print("\nSaved as additional_chexnet_features_with_folds.csv")



Saved as additional_chexnet_features_with_folds.csv
